<a href="https://colab.research.google.com/github/CallmeSharanya/BCI_Transfer_Learning/blob/EEGNET-%3ELSTM/EEGNet_%3ELSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne moabb torch torchvision torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.7/837.7 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 15.6 MB/s eta 0:00:00


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jscoderump/bci-competition-iv-dataset-2b")

print("Path to dataset files:", path)

100%|██████████| 217M/217M [00:05<00:00, 38.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jscoderump/bci-competition-iv-dataset-2b/versions/1


In [6]:
import os
dataset_path = "/root/.cache/kagglehub/datasets/jscoderump/bci-competition-iv-dataset-2b/versions/1"
print(os.listdir(dataset_path))

['B0401T.gdf', 'B0801T.gdf', 'B0402T.gdf', 'B0505E.gdf', 'B0103T.gdf', 'B0301T.gdf', 'B0604E.gdf', 'B0203T.gdf', 'B0502T.gdf', 'B0603T.gdf', 'B0304E.gdf', 'B0102T.gdf', 'B0101T.gdf', 'B0904E.gdf', 'B0105E.gdf', 'B0303T.gdf', 'B0405E.gdf', 'B0905E.gdf', 'B0704E.gdf', 'B0501T.gdf', 'B0605E.gdf', 'B0804E.gdf', 'B0803T.gdf', 'B0705E.gdf', 'B0902T.gdf', 'B0204E.gdf', 'B0201T.gdf', 'B0403T.gdf', 'B0805E.gdf', 'B0404E.gdf', 'B0205E.gdf', 'B0901T.gdf', 'B0305E.gdf', 'B0504E.gdf', 'B0302T.gdf', 'B0703T.gdf', 'B0202T.gdf', 'B0503T.gdf', 'B0602T.gdf', 'B0903T.gdf', 'B0701T.gdf', 'B0104E.gdf', 'B0802T.gdf', 'B0702T.gdf', 'B0601T.gdf']


In [ ]:
import pandas as pd

dataset_path="/root/.cache/kagglehub/datasets/jscoderump/bci-competition-iv-dataset-2b/versions/1"



In [15]:
from moabb.datasets.bnci import BNCI2014_004
from moabb.paradigms import MotorImagery

dataset = BNCI2014_004()

# Specify 2 classes (left vs right hand)
paradigm = MotorImagery(n_classes=2)

X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])

print(X.shape)
print(len(y))
print(metadata.head())

/usr/local/lib/python3.12/dist-packages/moabb/datasets/download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  0%|                                              | 0.00/34.2M [00:00<?, ?B/s]

SHA256 hash of downloaded file: 0da6e77ab0dab5b4aa1d2d5a6a542ac02f6768d3b7a76b0abe896ec1cf259919
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  0%|                                              | 0.00/18.6M [00:00<?, ?B/s]

SHA256 hash of downloaded file: 5effd365ae3733402286f2eea6b1ce482680a9f9ffc55c0b41bc63061a0161b5
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


(720, 3, 1126)
720
   subject session run
0        1  0train   0
1        1  0train   0
2        1  0train   0
3        1  0train   0
4        1  0train   0


In [16]:
import numpy as np
from sklearn.model_selection import train_test_split

# Normalize
X = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-6)

# Convert to (N, 1, C, T) for EEGNet
X = X[:, np.newaxis, :, :]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNet_LSTM(nn.Module):
    def __init__(self, n_classes, Chans=22, Samples=1000, dropoutRate=0.5):
        super().__init__()

        # -------- EEGNet Blocks --------
        self.firstconv = nn.Conv2d(1, 16, (1, 64), padding=(0, 32), bias=False)
        self.batchnorm1 = nn.BatchNorm2d(16)

        self.depthwiseConv = nn.Conv2d(
            16, 32, (Chans, 1), groups=16, bias=False
        )
        self.batchnorm2 = nn.BatchNorm2d(32)
        self.pooling1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropoutRate)

        self.separableConv = nn.Conv2d(
            32, 64, (1, 16), padding=(0, 8), bias=False
        )
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.pooling2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropoutRate)

        # -------- LSTM --------
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=2, batch_first=True)

        # -------- Classifier --------
        self.fc = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, 1, C, T)

        x = F.elu(self.batchnorm1(self.firstconv(x)))
        x = F.elu(self.batchnorm2(self.depthwiseConv(x)))
        x = self.pooling1(x)
        x = self.dropout1(x)

        x = F.elu(self.batchnorm3(self.separableConv(x)))
        x = self.pooling2(x)
        x = self.dropout2(x)

        # shape: (B, F, 1, T_reduced)
        x = x.squeeze(2)  # remove channel dim → (B, F, T)

        x = x.permute(0, 2, 1)  # → (B, T, F)

        x, _ = self.lstm(x)

        x = x[:, -1, :]  # last timestep
        x = self.fc(x)

        return x

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = EEGNet_LSTM(n_classes=4).to(device)
model = EEGNet_LSTM(n_classes=2, Chans=3, Samples=1000).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [20]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32)

In [37]:
for epoch in range(150):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ✅ Compute accuracy
        predicted = preds.argmax(dim=1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

    accuracy = correct / total

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}")

Epoch 1, Loss: 0.0270, Accuracy: 1.0000
Epoch 2, Loss: 0.1521, Accuracy: 0.9965
Epoch 3, Loss: 0.0768, Accuracy: 0.9983
Epoch 4, Loss: 0.0882, Accuracy: 0.9983
Epoch 5, Loss: 0.0404, Accuracy: 0.9983
Epoch 6, Loss: 0.0556, Accuracy: 0.9983
Epoch 7, Loss: 0.1339, Accuracy: 0.9965
Epoch 8, Loss: 0.3022, Accuracy: 0.9983
Epoch 9, Loss: 0.0270, Accuracy: 1.0000
Epoch 10, Loss: 0.3760, Accuracy: 0.9931
Epoch 11, Loss: 0.2939, Accuracy: 0.9948
Epoch 12, Loss: 0.2263, Accuracy: 0.9948
Epoch 13, Loss: 0.2809, Accuracy: 0.9948
Epoch 14, Loss: 0.0597, Accuracy: 0.9983
Epoch 15, Loss: 0.1411, Accuracy: 0.9983
Epoch 16, Loss: 0.1032, Accuracy: 0.9983
Epoch 17, Loss: 0.0387, Accuracy: 1.0000
Epoch 18, Loss: 0.1284, Accuracy: 0.9965
Epoch 19, Loss: 0.0648, Accuracy: 0.9983
Epoch 20, Loss: 0.1389, Accuracy: 0.9983
Epoch 21, Loss: 0.2218, Accuracy: 0.9931
Epoch 22, Loss: 0.0157, Accuracy: 1.0000
Epoch 23, Loss: 0.1262, Accuracy: 0.9965
Epoch 24, Loss: 0.0232, Accuracy: 1.0000
Epoch 25, Loss: 0.0063, A

In [38]:
from sklearn.metrics import accuracy_score

model.eval()
preds, true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)

        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(yb.numpy())

print("Test Accuracy:", accuracy_score(true, preds))

Test Accuracy: 0.7569444444444444
